# picoNewton_v4 — Google Colab workflow

Steps 1–4: source validation, standalone package tests and six-artery hydrodynamic reproduction. No membrane or Piezo1 coupling is executed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REPOSITORY = 'https://github.com/khalid-saqr/picoNewton.git'
BRANCH = 'agent/piconewton-v4-step4'
PROFILE = 'quick'  # change to 'publication' for N=150, Nt=2048, Nq=256
!rm -rf /content/picoNewton
!git clone --branch "$BRANCH" --single-branch "$REPOSITORY" /content/picoNewton
%cd /content/picoNewton
!python -m pip install -e './picoNewton_v4[dev]'


In [ ]:
from pathlib import Path
from piconewton_v4.runtime import create_runtime
from piconewton_v4.provenance import environment_record, write_json
drive_runs = Path('/content/drive/MyDrive/picoNewton_v4_runs')
runtime = create_runtime(drive_root=drive_runs, local_base=Path('/content/piconewton_v4_runtime'), metadata={'profile': PROFILE, 'branch': BRANCH, 'step': 4})
write_json(runtime.manifests / 'environment.json', environment_record(Path('/content/picoNewton')))
runtime.as_dict()


In [ ]:
import json
from piconewton_v4.sources import validate_cellml_sources
source_report = validate_cellml_sources(Path('/content/picoNewton/picoNewton_v4'))
print(json.dumps(source_report, indent=2))
assert source_report['ok'], source_report


In [ ]:
!pytest /content/picoNewton/picoNewton_v4/tests


In [ ]:
from piconewton_v4.workflow_step4 import run_step4
local_step4 = runtime.local_root / f'step4_{PROFILE}'
manifest = run_step4(package_root=Path('/content/picoNewton/picoNewton_v4'), output_root=local_step4, profile=PROFILE)
print(json.dumps(manifest['validation'], indent=2))
assert manifest['status'] == 'passed', manifest


In [ ]:
import shutil
persistent_step4 = runtime.source_data / f'step4_{PROFILE}'
shutil.copytree(local_step4, persistent_step4, dirs_exist_ok=False)
print(persistent_step4)


In [ ]:
import pandas as pd
summary = pd.read_csv(local_step4 / 'six_artery_hydrodynamic_summary.csv')
summary[['artery_name','mean_force_exposure_pn','peak_force_exposure_pn','mean_abs_wss_pa','peak_abs_wss_pa']]


In [ ]:
from datetime import datetime, timezone
write_json(runtime.manifests / 'step4_completion.json', {'step': 4, 'status': 'passed', 'completed_utc': datetime.now(timezone.utc).isoformat(), 'profile': PROFILE, 'output': str(persistent_step4)})
print(runtime.persistent_root)
